# RFX / Notes / Residual / 03 — Residual reversion (Avellaneda–Lee s-score, FX leg): cross-confirmation H1 ↔ H2

Protocol: [`RFX 001`](../../../../discussions/Rate_FX_Pairs/001_claude_rfx01_preregistration.md) §6. Reads `01/` (H1) and `02/` (H2) frozen candidates unchanged (SHA-256 checked).

- **Forward:** H1 candidates run on H2 (2021–2025). **Backward:** H2 candidates run on H1 (2016–2020).

| Gate | Condition |
|---|---|
| K1 | daily mean after spread (before commission) > 0, NW one-sided p, BH across the window's candidates, q ≤ 0.05 |
| K2 | daily net: block-bootstrap lower bound > 0 at blocks 10 and 20 days |
| K3 | daily net at 2× commission > 0 |
| K4 | daily net > 0 in ≥ 60% of counted years |

In [ ]:
STUDY = "RFX"
METHOD = "residual"

import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("RFX_SYNTHETIC") == "1"
if SYNTHETIC:                                   # never touch the real market-data folder in a smoke run
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("RFX_SYNTH_DIR") or tempfile.mkdtemp(prefix="rfx_")
    os.environ["RFX_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]
    os.environ.pop("DUKASCOPY_DIR", None)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F
from equity_trading import bond_seasonality as B
from equity_trading import bond_fx as X

cfg = X.BFXConfig()
if SYNTHETIC:
    cfg = replace(cfg, n_placebo=2, n_boot=2_000)
PAIRS = list(X.STUDIES[STUDY].pairings)
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 200)
SUF = "_synthetic" if SYNTHETIC else ""
OUT = Path.cwd() / ("03" + SUF); OUT.mkdir(exist_ok=True)
frozen = {}
for w, nn in (("H1", "01"), ("H2", "02")):
    c, s = X.load_frozen(Path.cwd() / (nn + SUF))
    frozen[w] = c
    print(f"{w}: {len(c)} frozen candidates (sha256 {s['candidates_sha256'][:12]}...), D0 = {s['D0']}: {', '.join(c['id']) or '-'}")

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"])
    if not (base / "Databento").exists():
        X.write_synthetic_rates(base, dt.date(2015, 12, 28), dt.date(2025, 12, 31), seed=5)

## 1. Found in both halves?

Pairings with a candidate in both H1 and H2, and whether the configuration is the same.

In [ ]:
both = frozen["H1"].merge(frozen["H2"], on="pairing", suffixes=(" H1", " H2"))[["pairing", "config H1", "config H2"]] if len(frozen["H1"]) and len(frozen["H2"]) else pd.DataFrame()
if len(both):
    both["same config"] = both["config H1"] == both["config H2"]
display(both if len(both) else "no pairing has a candidate in both halves"); both.to_csv(OUT / "03_found_in_both.csv", index=False)

## 2. Gates K1–K4

In [ ]:
results, trades, frames_by = [], {}, {}
for direction, (src, dst) in {"forward": ("H1", "H2"), "backward": ("H2", "H1")}.items():
    cand = frozen[src]
    if cand.empty:
        print(f"{direction}: no {src} candidates"); continue
    if dst not in frames_by:
        legs = {}
        frames_by[dst] = {p: X.frame(p, dst, cfg, legs, study=STUDY) for p in sorted(set(cand["pairing"]))}
    out, tr = X.confirm(cand, frames_by[dst], cfg)
    trades.update(tr)
    out = cand[["id", "pairing", "config", "mean_trade_net_bps"]].rename(columns={"mean_trade_net_bps": "disc_trade_net_bps"}).merge(out, on="id")
    out.insert(1, "test", direction); out.insert(2, "on", dst)
    out["verdict"] = np.where(out["verdict"] == "CONFIRMED", f"CONFIRMED_{direction.upper()}", "NOT_CONFIRMED")
    results.append(out)
res = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
if len(res):
    display(res.drop(columns=["boot_net_lower"]).round(3))
    print(res["verdict"].value_counts().to_string())
    res.to_csv(OUT / "03_cross_confirmation.csv", index=False)

## 3. Cumulative net on the confirmation window

In [ ]:
if len(res):
    n = len(res)
    fig, axes = plt.subplots(int(np.ceil(n / 2)), 2, figsize=(16, 3 * np.ceil(n / 2)), squeeze=False)
    for ax, (_, r) in zip(axes.flat, res.iterrows()):
        tr = trades[r["id"]].sort_values("exit_time")
        if len(tr):
            ax.plot(pd.to_datetime(tr["exit_time"]), tr["net_bps"].cumsum(), lw=0.9, color="C2" if r["verdict"].startswith("CONFIRMED") else "C3", label="net")
            ax.plot(pd.to_datetime(tr["exit_time"]), tr["mid_bps"].cumsum(), lw=0.7, color="grey", label="mid, no cost")
        ax.axhline(0, color="black", lw=0.5); ax.legend(fontsize=7)
        ax.set_title(f"{r['id']} {r['config']} on {r['on']}: {r['verdict']}", fontsize=9)
    for ax in axes.flat[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.savefig(OUT / "03_cumulative.png", dpi=110); plt.show()

In [ ]:
summary = {"study": f"RFX01-{STUDY}-cross", "method": METHOD, "bfx_version": X.BFX_VERSION, "synthetic": SYNTHETIC,
           "frozen": {w: c["id"].tolist() for w, c in frozen.items()}, "found_in_both": both.to_dict(orient="records") if len(both) else [],
           "confirmed": res.loc[res["verdict"].str.startswith("CONFIRMED"), ["id", "verdict"]].to_dict(orient="records") if len(res) else [],
           "results": res.drop(columns=["boot_net_lower"]).to_dict(orient="records") if len(res) else []}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps({k: summary[k] for k in ("method", "frozen", "confirmed")}, indent=2))